# 🧠 Notebook 07 — Baseline 3D SegResNet

---

## Objectives

This notebook validates the baseline segmentation model before starting training.

The notebook covers:

- Building SegResNet
- Inspecting the architecture
- Parameter analysis
- Dummy forward pass
- Loss function validation
- Optimizer setup
- GPU memory usage
- Saving and loading model weights

---

By the end of this notebook, the model will be fully verified and ready for training.

## 1. Setup

### 1.1 Install Dependencies

In [1]:
!pip -q install monai torchinfo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.5 MB/s eta 0:00:00


### 1.2 Imports

In [2]:
import torch
import torch.nn as nn

from monai.networks.nets import SegResNet
from monai.losses import DiceCELoss

from torchinfo import summary

import pandas as pd

2026-06-28 15:16:50.442202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782659810.666029      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782659810.724311      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782659811.211885      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782659811.211956      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782659811.211962      16 computation_placer.cc:177] computation placer alr

### 1.3 Device

In [3]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print(device)

cpu


## 2. Model Configuration

In [4]:
IN_CHANNELS = 4
OUT_CHANNELS = 4

ROI_SIZE = (128, 128, 128)

INIT_FILTERS = 32

BLOCKS_DOWN = (1, 2, 2, 4)

BLOCKS_UP = (1, 1, 1)

DROPOUT_PROB = 0.2

## 3. Build SegResNet

In [5]:
model = SegResNet(
    spatial_dims=3,
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
    init_filters=INIT_FILTERS,
    blocks_down=BLOCKS_DOWN,
    blocks_up=BLOCKS_UP,
    dropout_prob=DROPOUT_PROB,
).to(device)

print(model.__class__.__name__)

SegResNet


### 3.1 Parameter Count

In [6]:
total_params = sum(

    p.numel()

    for p in model.parameters()

)

trainable_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

pd.DataFrame({

    "Metric":[

        "Total Parameters",

        "Trainable Parameters",

    ],

    "Value":[

        total_params,

        trainable_params,

    ]

})

,Metric,Value
0,Total Parameters,18798660
1,Trainable Parameters,18798660


### 3.2 Architecture Summary (torchinfo)

In [7]:
summary(

    model,

    input_size=(

        1,

        4,

        128,

        128,

        128,

    ),

    depth=3,

)

Layer (type:depth-idx)                   Output Shape              Param #
SegResNet                                [1, 4, 128, 128, 128]     --
├─Convolution: 1-1                       [1, 32, 128, 128, 128]    --
│    └─Conv3d: 2-1                       [1, 32, 128, 128, 128]    3,456
├─Dropout3d: 1-2                         [1, 32, 128, 128, 128]    --
├─ModuleList: 1-3                        --                        --
│    └─Sequential: 2-2                   [1, 32, 128, 128, 128]    --
│    │    └─Identity: 3-1                [1, 32, 128, 128, 128]    --
│    │    └─ResBlock: 3-2                [1, 32, 128, 128, 128]    55,424
│    └─Sequential: 2-3                   [1, 64, 64, 64, 64]       --
│    │    └─Convolution: 3-3             [1, 64, 64, 64, 64]       55,296
│    │    └─ResBlock: 3-4                [1, 64, 64, 64, 64]       221,440
│    │    └─ResBlock: 3-5                [1, 64, 64, 64, 64]       221,440
│    └─Sequential: 2-4                   [1, 128, 32, 32, 32]   

## 4. Forward Pass Validation

In [8]:
dummy = torch.randn(

    1,

    4,

    128,

    128,

    128,

).to(device)

with torch.no_grad():

    output = model(dummy)

print(output.shape)

torch.Size([1, 4, 128, 128, 128])


### 4.1 Output Shape Assertion

In [9]:
assert output.shape == (

    1,

    4,

    128,

    128,

    128,

)

print("✅ Forward Pass Passed")

✅ Forward Pass Passed


## 5. Loss Function — DiceCELoss

In [10]:
from monai.losses import DiceCELoss

loss_function = DiceCELoss(

    to_onehot_y=True,

    softmax=True,

)

print(loss_function)

DiceCELoss(
  (dice): DiceLoss()
  (cross_entropy): CrossEntropyLoss()
  (binary_cross_entropy): BCEWithLogitsLoss()
)


### 5.1 Compute Loss on Dummy Data

In [11]:
dummy = torch.randn(
    1,
    4,
    128,
    128,
    128,
    device=device,
)

target = torch.randint(
    0,
    4,
    (
        1,
        1,
        128,
        128,
        128,
    ),
    device=device,
)

prediction = model(dummy)

loss = loss_function(
    prediction,
    target,
)

print(f"Loss: {loss.item():.4f}")

Loss: 2.1737


## 6. Optimizer — AdamW

In [12]:
optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=1e-4,

    weight_decay=1e-5,

)

print(type(optimizer).__name__)

AdamW


### 6.1 One Training Step

In [13]:
optimizer.zero_grad(set_to_none=True)

loss.backward()

optimizer.step()

print("✅ One Training Step Passed")

✅ One Training Step Passed


## 7. Mixed Precision (AMP)

In [14]:
from torch.amp import autocast

with autocast(
    device_type=device.type,
    enabled=device.type == "cuda",
):
    output = model(dummy)

print(output.shape)

torch.Size([1, 4, 128, 128, 128])


### 7.1 AMP Output Shape Assertion

In [15]:
assert output.shape == (
    1,
    4,
    128,
    128,
    128,
)

print("✅ AMP Forward Passed")

✅ AMP Forward Passed


## 8. Save & Load Checkpoint

In [16]:
from pathlib import Path

save_dir = Path("models")
save_dir.mkdir(exist_ok=True)

save_path = save_dir / "segresnet_baseline.pth"

torch.save(
    model.state_dict(),
    save_path,
)

print(save_path)

models/segresnet_baseline.pth


### 8.1 Load and Verify

In [17]:
loaded_model = SegResNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    init_filters=32,
)

loaded_model.load_state_dict(
    torch.load(
        save_path,
        map_location=device,
    )
)

loaded_model = loaded_model.to(device)

loaded_model.eval()

print("✅ Model Loaded Successfully")

✅ Model Loaded Successfully


### 8.2 Inference with Loaded Model

In [18]:
with torch.no_grad():

    loaded_output = loaded_model(dummy)

print(loaded_output.shape)

assert loaded_output.shape == (
    1,
    4,
    128,
    128,
    128,
)

print("✅ Loaded Model Forward Passed")

torch.Size([1, 4, 128, 128, 128])
✅ Loaded Model Forward Passed


## 9. Validation Summary Table

In [19]:
summary = pd.DataFrame({

    "Validation Step":[

        "Model Construction",

        "Architecture",

        "Parameter Count",

        "Forward Pass",

        "Loss Function",

        "Optimizer",

        "Backward Pass",

        "Mixed Precision",

        "Model Saving",

        "Model Loading",

    ],

    "Status":[

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

        "Passed",

    ]

})

summary

,Validation Step,Status
0,Model Construction,Passed
1,Architecture,Passed
2,Parameter Count,Passed
3,Forward Pass,Passed
4,Loss Function,Passed
5,Optimizer,Passed
6,Backward Pass,Passed
7,Mixed Precision,Passed
8,Model Saving,Passed
9,Model Loading,Passed


## 10. Final Output

In [20]:
print("=" * 60)
print("Notebook 07 Completed Successfully")
print("=" * 60)

print("✔ SegResNet verified")
print("✔ Forward pass verified")
print("✔ Loss verified")
print("✔ Optimizer verified")
print("✔ Backward pass verified")
print("✔ Mixed precision verified")
print("✔ Save / Load verified")

print()
print("Ready for Notebook 08 (Advanced Training)")

Notebook 07 Completed Successfully
✔ SegResNet verified
✔ Forward pass verified
✔ Loss verified
✔ Optimizer verified
✔ Backward pass verified
✔ Mixed precision verified
✔ Save / Load verified

Ready for Notebook 08 (Advanced Training)
